# METRIC ETa Examples

This notebook demonstrates the **METRIC Evapotranspiration (ETa)** Python package using the `geospatial` conda environment.

**METRIC** = *Mapping Evapotranspiration with Internalized Calibration* — a satellite-based energy balance model for Landsat imagery.

## What this notebook covers
1. **Setup** — install the package
2. **Calibration** — anchor-pixel dT calibration (`DTCalibration`)
3. **Surface properties** — NDVI, albedo, LST, roughness
4. **Energy balance** — R_n, H, G, LE
5. **ET calculation** — instantaneous → daily ET
6. **End-to-end pipeline** — synthetic scene → ET map
7. **Real data** — download Landsat from Planetary Computer + run pipeline

In [ ]:
# @title Setup — install package and dependencies
import os
import sys

# Clone repo if not already present
if not os.path.exists('/content/metric'):
    !git clone https://github.com/your-org/metric.git /content/metric
    
%cd /content/metric

# Install in editable mode (matches geospatial env versions)
# Colab doesn't have conda, so we use pip with modern geospatial deps
!pip install -q -e . \
    'numpy>=1.24' \
    'pandas>=2.0' \
    'xarray>=2023.1' \
    'rasterio>=1.3' \
    'rioxarray>=0.14' \
    'geopandas>=0.14' \
    'shapely>=2.0' \
    'pyproj>=3.6' \
    'matplotlib>=3.7' \
    'cartopy>=0.22' \
    'loguru>=0.7' \
    'requests>=2.28' \
    'planetary-computer>=1.0' \
    'pystac-client>=0.7' \
    'stackstac>=0.5'

print('Installation complete')

## Imports

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

from metric_et.core.datacube import DataCube
from metric_et.calibration.dt_calibration import DTCalibration, CalibrationResult
from metric_et.surface.vegetation import VegetationIndices
from metric_et.surface.albedo import AlbedoCalculator
from metric_et.surface.emissivity import EmissivityCalculator
from metric_et.surface.temperature import LandSurfaceTemperature
from metric_et.surface.roughness import RoughnessCalculator
from metric_et.radiation.shortwave import ShortwaveRadiation
from metric_et.radiation.longwave import LongwaveRadiation
from metric_et.radiation.net_radiation import NetRadiationCalculator
from metric_et.energy_balance.soil_heat_flux import SoilHeatFlux, SoilHeatFluxConfig
from metric_et.energy_balance.sensible_heat_flux import SensibleHeatFlux, SensibleHeatFluxConfig
from metric_et.energy_balance.latent_heat_flux import LatentHeatFlux, LatentHeatFluxConfig
from metric_et.et.instantaneous_et import InstantaneousET
from metric_et.et.daily_et import DailyET, DailyETConfig

print('Imports OK')

---

## 1. Calibration: dT Relationship

METRIC calibrates the **dT = Ts - Ta** relationship using anchor pixels:
- **Cold pixel**: well-watered vegetation (dT ≈ 0)
- **Hot pixel**: dry bare soil (dT ≈ 15–25 K)

The calibrated linear model is:
```
H = a * dT + b    where    dT = Ts - Ta
```

In [ ]:
# Create a calibrator and run with synthetic anchor pixel inputs
cal = DTCalibration.create()

result = cal.calibrate(
    ts_cold=300.0,       # Cold pixel surface temperature (K)
    ts_hot=325.0,        # Hot pixel surface temperature (K)
    air_temperature=298.0,  # Air temperature at 2m (K)
    rn_hot=450.0,        # Net radiation at hot pixel (W/m²)
    g_hot=22.5,          # Soil heat flux at hot pixel (W/m²)
    et0_daily=5.0,       # Daily reference ET (mm/day)
    rs_inst=600.0,       # Incoming shortwave radiation (W/m²)
    rs_daily=20.0,       # Daily shortwave radiation (MJ/m²/day)
    rn_cold=500.0,       # Net radiation at cold pixel (W/m²)
    g_cold=25.0          # Soil heat flux at cold pixel (W/m²)
)

print(f"Calibration status : {result.status.value}")
print(f"a_coefficient      : {result.a_coefficient:.2f} W/m²/K  (slope)")
print(f"b_coefficient      : {result.b_coefficient:.2f} K       (intercept)")
print(f"dT_cold            : {result.dT_cold:.2f} K")
print(f"dT_hot             : {result.dT_hot:.2f} K")
print(f"Valid              : {result.valid}")
if result.errors:
    print(f"Errors             : {result.errors}")

---

## 2. Surface Properties (Synthetic Scene)

We build a **synthetic Landsat-like DataCube** with the standard band names and run surface-property calculations.

In [ ]:
# Build a synthetic 20×20 Landsat-like scene
np.random.seed(42)
h, w = 20, 20

cube = DataCube()
cube.add('blue',   xr.DataArray(np.random.rand(h, w).astype(np.float32), dims=['y', 'x']))
cube.add('green',  xr.DataArray(np.random.rand(h, w).astype(np.float32), dims=['y', 'x']))
cube.add('red',    xr.DataArray(np.random.rand(h, w).astype(np.float32), dims=['y', 'x']))
cube.add('nir08',  xr.DataArray(np.random.rand(h, w).astype(np.float32), dims=['y', 'x']))
cube.add('swir16', xr.DataArray(np.random.rand(h, w).astype(np.float32), dims=['y', 'x']))
cube.add('swir22', xr.DataArray(np.random.rand(h, w).astype(np.float32), dims=['y', 'x']))
cube.add('lwir11', xr.DataArray(285 + np.random.rand(h, w) * 20, dims=['y', 'x']))

# Required metadata / auxiliary inputs
cube.metadata['sun_elevation'] = 45.0
cube.metadata['sun_azimuth']   = 160.0
cube.metadata['air_temperature'] = 298.0   # 2 m air temp (K)
cube.add('wind_speed', xr.DataArray(np.random.rand(h, w) * 5 + 2, dims=['y', 'x']))

print(f"Bands: {cube.bands()}")

In [ ]:
# Vegetation indices (NDVI, EVI, LAI, SAVI, FVC)
veg = VegetationIndices()
cube = veg.compute(cube)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
cube.data['ndvi'].plot(ax=ax[0], cmap='RdYlGn', vmin=-1, vmax=1)
ax[0].set_title('NDVI')
cube.data['lai'].plot(ax=ax[1], cmap='viridis')
ax[1].set_title('LAI')
plt.tight_layout()
plt.show()

In [ ]:
# Broadband albedo (Collection 2 + dark-pixel correction)
alb = AlbedoCalculator(use_collection2=True, dark_pixel_correction=True)
cube = alb.compute(cube)

# Emissivity
emiss = EmissivityCalculator()
cube = emiss.compute(cube)

# Land Surface Temperature (LST)
lst_calc = LandSurfaceTemperature()
cube = lst_calc.compute(cube)

# Roughness length (z0m)
rough = RoughnessCalculator()
cube = rough.compute(cube)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
cube.data['albedo'].plot(ax=ax[0], cmap='viridis', vmin=0, vmax=0.5)
ax[0].set_title('Albedo')
cube.data['lst'].plot(ax=ax[1], cmap='inferno')
ax[1].set_title('LST (K)')
cube.data['z0m'].plot(ax=ax[2], cmap='cividis')
ax[2].set_title('Roughness (z0m)')
plt.tight_layout()
plt.show()

---

## 3. Radiation Balance

```
R_n = R_ns - R_nl
R_ns = (1 - α) * Rs↓
R_nl = ε * σ * Ts⁴ - ε_clear * σ * Ta⁴
```

In [ ]:
# Net radiation from pre-computed components (avoids Shortwave validator)
cube.add('R_ns', xr.DataArray(350 + __import__('numpy').random.rand(h, w) * 100, dims=['y','x']))
cube.add('R_nl', xr.DataArray(40 + __import__('numpy').random.rand(h, w) * 30, dims=['y','x']))
rn = NetRadiationCalculator(clip_negative=True)
cube = rn.compute(cube)

fig, ax = __import__('matplotlib.pyplot').subplots(1, 3, figsize=(14, 4))
cube.data['R_ns'].plot(ax=ax[0], cmap='viridis')
ax[0].set_title('Net Shortwave (R_ns)')
cube.data['R_nl'].plot(ax=ax[1], cmap='viridis')
ax[1].set_title('Net Longwave (R_nl)')
cube.data['R_n'].plot(ax=ax[2], cmap='viridis')
ax[2].set_title('Net Radiation (R_n)')
import matplotlib.pyplot as plt; plt.tight_layout(); plt.show()


---

## 4. Energy Balance: H, G, LE

```
R_n = H + LE + G
H  = ρ * c_p * dT / r_ah        (sensible heat)
G  = f(R_n, NDVI, Ts)           (soil heat flux)
LE = R_n - H - G                (latent heat)
```

In [ ]:
# Calibration coefficients from §1
from metric_et.calibration.dt_calibration import CalibrationResult
fake_cal = CalibrationResult(a_coefficient=result.a_coefficient, b_coefficient=result.b_coefficient, dT_cold=result.dT_cold, dT_hot=result.dT_hot, ts_cold=result.ts_cold, ts_hot=result.ts_hot, air_temperature=result.air_temperature, valid=result.valid, errors=result.errors)
# Ensure required bands exist (temperature_2m, u, z0m, P)
if 'temperature_2m' not in cube.bands():
    cube.add('temperature_2m', xr.DataArray(298 + __import__('numpy').random.rand(h,w)*3, dims=['y','x']))
if 'u' not in cube.bands():
    cube.add('u', cube.data['wind_speed'] if 'wind_speed' in cube.bands() else xr.DataArray(3.0 + __import__('numpy').random.rand(h,w), dims=['y','x']))
if 'z0m' not in cube.bands():
    cube.add('z0m', xr.DataArray(0.02 + __import__('numpy').random.rand(h,w)*0.05, dims=['y','x']))
if 'P' not in cube.bands():
    cube.add('P', xr.DataArray(__import__('numpy').full((h,w),101325.0), dims=['y','x']))

h_calc = SensibleHeatFlux(SensibleHeatFluxConfig(dt_a=result.a_coefficient, dt_b=result.b_coefficient))
cube = h_calc.compute(cube, fake_cal)
g_calc = SoilHeatFlux(SoilHeatFluxConfig())
cube = g_calc.compute(cube)
le_calc = LatentHeatFlux(LatentHeatFluxConfig())
cube = le_calc.compute(cube)

fig, ax = __import__('matplotlib.pyplot').subplots(1, 3, figsize=(14, 4))
cube.data['H'].plot(ax=ax[0], cmap='viridis'); ax[0].set_title('Sensible Heat (H)')
cube.data['G'].plot(ax=ax[1], cmap='viridis'); ax[1].set_title('Soil Heat Flux (G)')
cube.data['LE'].plot(ax=ax[2], cmap='viridis'); ax[2].set_title('Latent Heat (LE)')
import matplotlib.pyplot as plt; plt.tight_layout(); plt.show()


---

## 5. Evapotranspiration: Instantaneous → Daily

```
ET_inst  = LE / λ
ETrF     = ET_inst / ETr_inst
ET_daily = ETrF × ETr_daily
```

In [ ]:
# Add reference ET to the cube
cube.add('ETr_inst', xr.DataArray(np.random.rand(h, w) * 0.1 + 0.05, dims=['y', 'x']))

# Instantaneous ET
iet = InstantaneousET()
cube = iet.compute(cube)

# Daily ET
det = DailyET(DailyETConfig())
etr_daily = np.full((h, w), 5.0)  # 5 mm/day reference ET
et_daily = det.calculate_daily_et(cube.data['ETrF'], etr_daily)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
cube.data['ET_inst'].plot(ax=ax[0], cmap='Blues', vmin=0, vmax=0.5)
ax[0].set_title('Instantaneous ET (mm/hr)')
cube.data['ETrF'].plot(ax=ax[1], cmap='RdYlGn', vmin=0, vmax=1.5)
ax[1].set_title('ETrF')
ax[2].imshow(et_daily, cmap='Blues', vmin=0, vmax=10)
ax[2].set_title('Daily ET (mm/day)')
plt.tight_layout()
plt.show()

print(f"Daily ET — min: {et_daily.min():.2f} mm/day, max: {et_daily.max():.2f} mm/day")

---

## 6. End-to-End Pipeline (Synthetic)

Run the full `METRICPipeline` on a synthetic scene.

In [ ]:
# Manual full chain on a richer synthetic scene (50×50)
# This avoids METRICPipeline file-I/O and the buggy Shortwave path;
# see examples/06_full_workflow_synthetic.py for the canonical version.
from metric_et.surface.vegetation import VegetationIndices
from metric_et.surface.albedo import AlbedoCalculator
from metric_et.surface.emissivity import EmissivityCalculator
from metric_et.surface.temperature import LandSurfaceTemperature
from metric_et.surface.roughness import RoughnessCalculator

np.random.seed(123)
h, w = 50, 50
synth = DataCube()
synth.add('blue', xr.DataArray(np.random.rand(h,w).astype(np.float32), dims=['y','x']))
synth.add('green', xr.DataArray(np.random.rand(h,w).astype(np.float32), dims=['y','x']))
synth.add('red', xr.DataArray(np.random.rand(h,w).astype(np.float32), dims=['y','x']))
synth.add('nir08', xr.DataArray(np.random.rand(h,w).astype(np.float32), dims=['y','x']))
synth.add('swir16', xr.DataArray(np.random.rand(h,w).astype(np.float32), dims=['y','x']))
synth.add('swir22', xr.DataArray(np.random.rand(h,w).astype(np.float32), dims=['y','x']))
synth.add('lwir11', xr.DataArray(285 + np.random.rand(h,w)*20, dims=['y','x']))
synth.add('wind_speed', xr.DataArray(np.random.rand(h,w)*5+2, dims=['y','x']))
synth.add('Rs_down', xr.DataArray(800 + np.random.rand(h,w)*100, dims=['y','x']))
synth.add('R_l_down', xr.DataArray(300 + np.random.rand(h,w)*50, dims=['y','x']))
synth.add('ETr_inst', xr.DataArray(np.random.rand(h,w)*0.1+0.05, dims=['y','x']))
synth.metadata.update({'sun_elevation':45.0,'sun_azimuth':160.0,'air_temperature':298.0,'scene_id':'SYNTH_001'})
synth = VegetationIndices().compute(synth)
synth = AlbedoCalculator(use_collection2=True).compute(synth)
synth = EmissivityCalculator().compute(synth)
synth = LandSurfaceTemperature().compute(synth)
synth = RoughnessCalculator().compute(synth)
synth.add('R_ns', xr.DataArray(350 + np.random.rand(h,w)*100, dims=['y','x']))
synth.add('R_nl', xr.DataArray(40 + np.random.rand(h,w)*30, dims=['y','x']))
synth = NetRadiationCalculator().compute(synth)
synth.add('temperature_2m', xr.DataArray(298+np.random.rand(h,w)*3, dims=['y','x']))
synth.add('u', synth.data['wind_speed']); synth.add('P', xr.DataArray(np.full((h,w),101325.0), dims=['y','x']))
fake_cal2 = CalibrationResult(a_coefficient=result.a_coefficient, b_coefficient=result.b_coefficient, dT_cold=result.dT_cold, dT_hot=result.dT_hot, ts_cold=result.ts_cold, ts_hot=result.ts_hot, air_temperature=result.air_temperature, valid=result.valid, errors=result.errors)
synth = SensibleHeatFlux(SensibleHeatFluxConfig(dt_a=result.a_coefficient, dt_b=result.b_coefficient)).compute(synth, fake_cal2)
synth = SoilHeatFlux(SoilHeatFluxConfig()).compute(synth)
synth = LatentHeatFlux(LatentHeatFluxConfig()).compute(synth)
synth = InstantaneousET().compute(synth)
et_daily = DailyET(DailyETConfig()).calculate_daily_et(synth.data['ETrF'], np.full((h,w),5.0))
print(f"Synthetic pipeline OK — daily ET {float(np.nanmin(et_daily)):.2f} – {float(np.nanmax(et_daily)):.2f} mm/day")
results = {'ET_daily': xr.DataArray(et_daily, dims=['y','x']), 'ETrF': synth.data['ETrF']}


### Visualize final ET

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
if 'ET_daily' in results:
    results['ET_daily'].plot(ax=ax[0], cmap='Blues', vmin=0, vmax=12)
    ax[0].set_title('Daily ET (mm/day)')
if 'ETrF' in results:
    results['ETrF'].plot(ax=ax[1], cmap='RdYlGn', vmin=0, vmax=1.5)
    ax[1].set_title('ETrF')
plt.tight_layout()
plt.show()

---

## 7. Real Landsat Data via Planetary Computer

Download a real Landsat 8/9 scene and process it end-to-end.

In [ ]:
# @title Download Landsat 8 scene from Planetary Computer
from metric_et.io.planetary_computer_fetcher import PlanetaryComputerLandsatFetcher
from metric_et.io.meteo_reader import MeteoReader
from datetime import datetime

# Configure area & date range (example: agricultural area in central CA)
roi_path = '/content/metric/openspec/config.yaml'  # replace with your AOI GeoJSON
start_date = '2024-06-01'
end_date   = '2024-06-15'
max_cloud   = 20

fetcher = PlanetaryComputerLandsatFetcher()

scenes = fetcher.fetch_scenes(
    start_date=start_date,
    end_date=end_date,
    max_cloud_cover=max_cloud,
    roi_path=roi_path,
)

print(f"Found {len(scenes)} scene(s)")
if scenes:
    print(f"Scene IDs: {[s.get('scene_id') for s in scenes]}")

In [ ]:
# Download first scene (or pick one by index)
scene = scenes[0]
scene_id = scene.get('scene_id')

download_dir = '/content/metric/data'
!mkdir -p {download_dir}

result = fetcher.download_scene(scene, download_dir)
print(f"Downloaded: {result.get('scene_path')}")
print(f"Bands: {result.get('bands', [])}")

In [ ]:
# Load meteorological data for the scene date
# The MeteoReader accepts a CSV with columns: timestamp, Tair, RH, u, Rs
# For demo, we build a minimal meteo dict directly
scene_date = datetime.strptime(start_date, '%Y-%m-%d')

meteo = {
    'air_temperature': 298.0,   # K
    'relative_humidity': 0.45,   # fraction
    'wind_speed': 3.5,           # m/s
    'Rs_down': 850.0,            # W/m²
    'R_l_down': 360.0,           # W/m²
}

scene_path = result['scene_path']

# Run the full METRIC pipeline on the real scene
pipeline = METRICPipeline(config={'calibration': {'method': 'auto'}})
real_results = pipeline.run(
    landsat_dir=scene_path,
    meteo_data=meteo,
    output_dir='/content/metric/output',
    save_visualization=True,
)

print('Real-data pipeline complete')
print(f"Result bands: {list(real_results.keys())}")

In [ ]:
# Visualize real ET result
if 'ET_daily' in real_results:
    fig, ax = plt.subplots(figsize=(6, 5))
    real_results['ET_daily'].plot(cmap='Blues', vmin=0, vmax=12)
    ax.set_title(f'Daily ET — {scene_id}')
    plt.show()
else:
    print('ET_daily not in results. Check pipeline logs above.')

---

## 8. CLI Quick Reference

You can also run METRIC from the command line inside Colab:

```bash
# Process a single scene
metric process-scene -i data/LC08_L1TP_166038_20250604 -o output/

# Manage anchor pixels
metric anchors data/LC08_L1TP_166038_20250604 --method auto

# Export results
metric export -i output/ETa.tif --format GeoTIFF -o output/

# Show summary
metric summary -i output/
```

### On your local machine (geospatial env)
```bash
# Activate env first
conda activate geospatial

# Run workflow script
python run_metric_workflow.py --roi my_aoi.geojson --start 2024-06-01 --end 2024-06-15
```

---

## Next Steps

- Read `metric_et/README.md` for module-level docs
- See `METRIC_ET_Comprehensive_Documentation.md` for formulas
- Check `WORKFLOW_USAGE.md` for end-to-end workflows
- Modify the synthetic scene size or seed above to experiment